In [ ]:
import os, glob, shutil, subprocess
WORKDIR = '/kaggle/working'
DATASET_ZIP = '/kaggle/input/smartmarl-codebase/smartmarl_kaggle.zip'
os.makedirs('/kaggle/working/output', exist_ok=True)
os.makedirs('/kaggle/working/results/raw', exist_ok=True)
subprocess.run(['apt-get', 'update', '-q'], check=False)
subprocess.run(['apt-get', 'install', '-y', 'sumo', 'sumo-tools'], check=True)
os.environ['SUMO_HOME'] = '/usr/share/sumo'
subprocess.run(['unzip', '-q', '-o', DATASET_ZIP, '-d', WORKDIR], check=True)


In [ ]:
smoke = subprocess.run(
  ['python', 'train.py', '--scenario', 'standard', '--ablation', 'full', '--seed', '0', '--episodes', '1'],
  cwd=WORKDIR,
  capture_output=True,
  text=True,
)
print(smoke.stdout[-2500:])
assert 'Mock mode: False' in smoke.stdout, 'Mock mode detected; stop.'


In [ ]:
variant = 'full'
seed_start = 21
seed_end = 29

for seed in range(seed_start, seed_end + 1):
    result_name = f'standard_{variant}_seed{seed}.json'
    cmd = [
        'python', 'train.py',
        '--scenario', 'standard',
        '--ablation', variant,
        '--seed', str(seed),
        '--episodes', '3000',
        '--steps_per_episode', '300',
        '--checkpoint_every', '100',
        '--resume',
        '--skip_existing',
        '--result_json', f'results/raw/{result_name}',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORKDIR, check=True)

for f in glob.glob('/kaggle/working/results/raw/*.json'):
    shutil.copy(f, '/kaggle/working/output/' + os.path.basename(f))
print('Done.')
